<a href="https://colab.research.google.com/github/Perennial123/economics-notes-UW-ECON201/blob/main/uwAnalyticsAPIKey.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Git, NLP & LLMs — Workshop Notebook
### Welcome! You're going to build with AI today.

---

This notebook is split into **4 sections**:

| # | Section | Time |
|---|---------|------|
| 1 | Setup — installing & connecting to the API | ~3 min |
| 2 | Your First LLM Call | ~5 min |
| 3 | Prompt Engineering | ~5 min |
| 4 | NLP Tasks (Sentiment, Summarization, Code Gen) | ~5 min |
| 5 | Free Play — try anything! | ~5 min |

---

> **How to run a cell:** Click on it, then press **Shift + Enter** (or click the play button on the left)
>
> **If something breaks:** Don't panic — read the error message. Most errors are typos or missing quotes. Ask for help!

---
## Section 1: Setup

Before we can talk to an LLM, we need to:
1. **Install** the OpenAI Python package (OpenRouter uses the same interface — no extra library needed!)
2. **Set your OpenRouter API key** so the model knows you're allowed to use it

> **What is OpenRouter?** It's a service that gives you access to many different AI models (GPT, Claude, Llama, etc.) through a single API. Think of it like a universal remote for LLMs.
>
> **Where to get a key:** Sign up at [openrouter.ai](https://openrouter.ai) → go to **Keys** in your dashboard → create a new key.
>
> Think of the API key like a password — it's what lets you send requests to the model. Never share it publicly!

In [5]:
# Step 1: Install the OpenAI library
# OpenRouter is compatible with the OpenAI Python library — no separate package needed!
# The exclamation mark (!) means we're running a terminal command inside this notebook
!pip install openai --quiet

print("OpenAI library installed! (Works with OpenRouter too)")

OpenAI library installed! (Works with OpenRouter too)


In [17]:
# Step 2: Set your OpenRouter API key
# We store it as an environment variable so it's not hardcoded in your script

import os

# -------------------------------------------------------------------
# PASTE YOUR OPENROUTER API KEY BETWEEN THE QUOTES BELOW
#    Get one free at: https://openrouter.ai → Dashboard → Keys
# -------------------------------------------------------------------
os.environ["OPENROUTER_API_KEY"] = "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"
# -------------------------------------------------------------------

print("OpenRouter API key set! (Keep this secret — never share it publicly)")

OpenRouter API key set! (Keep this secret — never share it publicly)


In [8]:
# Step 3: Create the client — pointed at OpenRouter instead of OpenAI
# OpenRouter uses the same interface as OpenAI, but with a different base URL
# That's the only difference! All the code after this stays exactly the same.

from openai import OpenAI

client = OpenAI(
    api_key=os.environ["OPENROUTER_API_KEY"],
    base_url="https://openrouter.ai/api/v1",  # ← this points to OpenRouter
)

print("Client ready! Connected to OpenRouter — you're all set.")

Client ready! Connected to OpenRouter — you're all set.


---
## Section 2: Your First LLM Call

Let's send our first message to the model — just like texting it.

Here's what the code does:
- `model` → which AI model to use. On OpenRouter, models use the format `provider/model-name`
- `messages` → the conversation history. We send a list with our message.
- `role: "user"` → that's us talking
- `content` → the actual text we're sending

> We're using `openrouter/auto` — this is OpenRouter's **free** model tier, so it costs nothing to run. OpenRouter will automatically route your request to a capable free model. You can also access Claude, Llama, Gemini, and many others just by swapping the model name!

In [18]:
# Your first LLM call!

response = client.chat.completions.create(
    model="openrouter/auto",
    messages=[
        {"role": "user", "content": "What model are you"}
    ]
)

# Extract the text response from the returned object
answer = response.choices[0].message.content

print("Model says:")
print("-" * 40)
print(answer)

Model says:
----------------------------------------
 I don't have a physical form or model. I'm a text-based artificial intelligence designed to help answer questions and provide information. I don't have the ability to exist in the physical world.


### Breaking down the code

```python
response.choices[0].message.content
```

The model returns a **response object** (not just plain text). Think of it like a package:
- `response.choices` → list of possible responses (usually just one)
- `[0]` → we grab the first one
- `.message.content` → the actual text inside

> Try it: Change `"Explain AI in simple terms"` to something else and run the cell again!

---
## Section 3: Prompt Engineering

**Prompt engineering** = the skill of writing better instructions to get better outputs.

The model doesn't "think" — it predicts what text should come next based on your input.
So **how you phrase your prompt dramatically changes the output.**

Let's see this in action with a side-by-side comparison.

In [ ]:
# Helper function to keep our code clean
# (A function is a reusable block of code — we define it once, use it many times)

def ask(prompt):
    """Send a prompt to the model and return the response text."""
    response = client.chat.completions.create(
        model="openrouter/auto",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

print("Helper function defined — we'll use ask() from now on!")

In [ ]:
# Vague prompt — minimal context
bad_prompt = "Explain AI"

print(f"Prompt: '{bad_prompt}'")
print("-" * 40)
print(ask(bad_prompt))

In [ ]:
# Specific prompt — clear audience, format, and context
good_prompt = "Explain AI to a 10-year-old using a fun analogy. Keep it under 5 sentences."

print(f"Prompt: '{good_prompt}'")
print("-" * 40)
print(ask(good_prompt))

### Prompt Engineering Tips

| Tip | Example |
|-----|---------|
| **Specify the audience** | "Explain to a beginner..." |
| **Give a format** | "Give me a bullet list of..." |
| **Set a length** | "In under 3 sentences..." |
| **Give a role** | "You are a helpful tutor. Explain..." |
| **Show an example** | "Like this: [example]. Now do it for..." |

> Try it: Edit the `good_prompt` above. What happens if you say "Explain AI to a pirate"? Or ask for a numbered list?

In [ ]:
# BONUS: System prompts — giving the model a "role" before the conversation
# The "system" role sets the model's personality/behavior

response = client.chat.completions.create(
    model="openrouter/auto",
    messages=[
        {"role": "system", "content": "You are a witty professor who explains everything using sports analogies."},
        {"role": "user",   "content": "What is machine learning?"}
    ]
)

print("Professor Mode:")
print("-" * 40)
print(response.choices[0].message.content)

---
## Section 4: NLP Tasks

**Natural Language Processing (NLP)** = using computers to understand and work with human language.

LLMs are incredibly powerful for NLP tasks because they've been trained on massive amounts of text.
Let's try a few classic NLP tasks using nothing but prompts!

### Task 1: Sentiment Analysis
Sentiment analysis = figuring out if text is **positive, negative, or neutral**.

Old approach: You'd write hundreds of rules or train a custom ML model.
New approach: Just ask!

In [ ]:
# Sentiment Analysis

def classify_sentiment(text):
    prompt = f"""Classify the sentiment of the following text as POSITIVE, NEGATIVE, or NEUTRAL.
Only respond with one word.

Text: "{text}"
Sentiment:"""
    return ask(prompt)

# Test it out!
examples = [
    "I love this workshop, it's so interesting!",
    "This is the worst coffee I've ever had.",
    "The meeting is scheduled for 3pm tomorrow."
]

for text in examples:
    sentiment = classify_sentiment(text)
    print(f"Text: '{text}'")
    print(f"Sentiment: {sentiment.strip()}")
    print()

In [ ]:
# Try your own!
# Change this text to anything you want and see what the model says

my_text = "I'm not sure how I feel about this..."  # ← change me!

print(f"Text: '{my_text}'")
print(f"Sentiment: {classify_sentiment(my_text).strip()}")

### Task 2: Summarization
Summarization = taking a long piece of text and extracting the key points.

Super useful for: research papers, news articles, long emails, documentation, etc.

In [ ]:
# Summarization

long_text = """
Git is a distributed version control system created by Linus Torvalds in 2005.
It was designed to handle everything from small to very large projects with speed and efficiency.
Git tracks changes in any set of files, usually used for coordinating work among programmers who
are collaboratively developing source code during software development. Its goals include speed,
data integrity, and support for distributed, non-linear workflows. Git gives each developer
a local copy of the full development history, and changes are copied from one such repository
to another. Every Git directory on every computer is a full-fledged repository with complete
history and full version-tracking abilities.
"""

summary_prompt = f"""Summarize the following text in 2 bullet points. Keep it simple and clear.

Text:
{long_text}
"""

print("📄 Original text length:", len(long_text.split()), "words")
print()
print("📋 Summary:")
print(ask(summary_prompt))

### Task 3: Code Generation
LLMs are surprisingly good at writing code — they were trained on massive amounts of GitHub repositories.

You can describe what you want in plain English and get working code back.

In [ ]:
# Code Generation

code_prompt = """
Write a Python function that:
- Takes a list of numbers as input
- Returns the average (mean) of those numbers
- Handles the case where the list is empty (return 0)

Include a simple example showing how to use it.
Only return the code, no explanation.
"""

generated_code = ask(code_prompt)

print("Generated Code:")
print("-" * 40)
print(generated_code)

In [ ]:
# Now let's actually RUN the generated code!
# Copy the function from above and paste it here to test it

# Paste the generated function here:
# (example below — replace with whatever the model gave you)

def calculate_average(numbers):
    if len(numbers) == 0:
        return 0
    return sum(numbers) / len(numbers)

# Test it!
print(calculate_average([10, 20, 30, 40, 50]))  # Should print 30.0
print(calculate_average([]))                     # Should print 0

---
## Section 5: Free Play!

This is your sandbox. Try anything you want.

Here are some fun ideas to get you started — but don't let these limit you!

In [ ]:
# Write a rap about Git

print(ask("Write a short rap (8 lines) about Git version control. Make it fun and educational."))

In [ ]:
# Explain something complex in simple terms

topic = "neural networks"  # ← change to any topic!

print(ask(f"Explain {topic} like I'm 12 years old, using a real-world analogy."))

In [ ]:
# Q&A — ask it anything

question = "What should I learn first if I want to get into AI?"  # ← change to your question!

print(ask(question))

In [ ]:
# BONUS: Multi-turn conversation (simulating a real chatbot)
# In a real chatbot, we keep track of the whole conversation history
# and send it with every new message

conversation_history = [
    {"role": "system", "content": "You are a friendly AI tutor helping students learn about programming and AI. Keep responses short and encouraging."}
]

def chat(user_message):
    """Add a message to the conversation and get a response."""
    conversation_history.append({"role": "user", "content": user_message})

    response = client.chat.completions.create(
        model="openrouter/auto",
        messages=conversation_history
    )

    assistant_message = response.choices[0].message.content
    conversation_history.append({"role": "assistant", "content": assistant_message})

    return assistant_message

# Start a conversation!
print("You: What's the difference between AI and ML?")
print(f"AI: {chat('What is the difference between AI and ML?')}")
print()
print("You: Can you give me an example of ML in real life?")
print(f"AI: {chat('Can you give me an example of that in real life?')}")

In [ ]:
# YOUR SANDBOX — write anything you want here!
# Try crazy prompts, break things, experiment

my_prompt = """Write a haiku about debugging code at 2am."""

print(ask(my_prompt))

---
## Wrap-Up — What You Just Built

You've officially used a real LLM API to:

Make your first API call — talked to GPT-4.1-mini via OpenRouter  
Engineered better prompts — saw how phrasing changes everything  
Built NLP tools — sentiment analysis, summarization, code generation  
Built a multi-turn chatbot — with memory of past messages  

---

### What can you build next?

With just what you learned today + a bit more time:

| Project | What you'd need |
|---------|----------------|
| Study assistant chatbot | Multi-turn chat + a good system prompt |
| Resume reviewer | Feed in your resume text, ask for feedback |
| News summarizer | Grab article text, ask for a summary |
| Discord/Slack bot | Connect this API to a bot framework |
| Your startup idea | Literally any text task you want to automate |

---

### Resources to keep learning

- [OpenRouter Docs](https://openrouter.ai/docs) — models, pricing, and API reference
- [OpenRouter Model List](https://openrouter.ai/models) — browse all available models
- [Prompt Engineering Guide](https://www.promptingguide.ai/) — free, comprehensive
- [fast.ai](https://www.fast.ai/) — free practical deep learning course
- [Git docs](https://git-scm.com/doc) — official Git reference

---

> "The best way to learn AI is to build something with it."  
> — Start with something small. Ship it. Iterate.